# 언어모델 & 생성모델

## NLP

NLP(자연어처리): 사람이 쓰는 말(자연어)을 컴퓨터가 분석, 이해, 생성할 수 있게 하는 기술

- 자연어 vs 프로그래밍 언어
  - 자연어: 유연한데 모호함
  - 프로그래밍 언어: 명확하고 엄격함. 일대일 대응

- 자연어가 컴퓨터한테 어려운 이유 (특징)
  - 이산적: 단어, 문자로 뚝뚝 끊어져 있음 → 그래서 벡터로 바꿔야 함 (임베딩)
  - 순차적: 순서 바뀌면 의미 바뀜 → RNN, 트랜스포머 같은 구조 필요
  - 중의성: "배" = 신체? 과일? 선박? → 문맥 전체를 봐야 함
  - 희소성: 단어 조합이 너무 많아서 다 학습 x → 안 본 표현도 확률적으로 추론해야 함
  - 가변성: 신조어, 줄임말 계속 생김 → 규칙 말고 데이터로 배워야 함

- NLP가 하는 일
  1. 형태소 분석: 의미 있는 제일 작은 단위로 쪼갬 (나/는/학교/에/간다)
  2. 구문 분석: 문법 구조 파악 (주어, 목적어, 서술어)
  3. 의미 분석: "은행에서 돈 찾았다" → 은행 = 금융기관
  4. 담화 분석: 여러 문장 사이 관계 ("그는" = 철수)

### 언어 모델이란?
- 텍스트 잔뜩 학습해서 다음 단어 등장 확률 예측 + 문장이 자연스러운지 판단하는 모델
- 걍 사람이 말하는 방식 따라하는 시스템
- "나는 아침에 밥을 ___" → 먹었다 80%, 마셨다 15%, 달렸다 5%
  - 확률 높은 문장 = 자연스러운 문장
- 학습 방식에 따라 **통계적 언어모델(SLM)** / **신경망 언어모델(NLM)** 으로 나뉨

# 통계 기반 모델

### SLM과 분포 가설

**분포 가설: 비슷한 문맥에서 나오는 단어는 의미도 비슷할 거다**
- 단어 뜻은 걔가 누구랑 같이 나오는지 보면 앎
- "아침에 커피를 마셨다" / "아침에 차를 마셨다" → 커피랑 차는 비슷한 애들
- 나중에 임베딩의 이론적 기반이 됨!

SLM은 이 아이디어로 확률 계산함
- 조건부 확률: P(갔다 | 나는 학교에) → 앞 문맥이 주어졌을 때 다음 단어 확률
- 카운트 기반 확률: "나는 학교에"가 100번 나왔는데 그 뒤에 갔다 70번 → 70%

- 한계: **희소 문제**
  - 학습 데이터에 한 번도 안 나온 조합은 확률 0 때려버림
  - "나는 학교에 순간이동했다" → 문법상 ㄱㄴ인데 데이터에 없으면 0

### N-gram

- 왜 나옴?
  - SLM은 앞에 나온 단어 **전부** 조건으로 봄
  - 문장 길어지면 → 조합 폭발해서 희소성 더 심해짐 + 계산, 저장 너무 무거움
  - 그래서 "걍 최근 N-1개만 보자" 해서 나온 게 N-gram

- 정의: 연속된 N개 단어 묶음. 직전 N-1개 단어로 다음 단어 예측
  - 4-gram이면 "An adorable little boy is spreading ___"에서 boy, is, spreading 3개만 봄
  - An adorable little은 버림ㅠ
- SLM이랑 별개 개념 x. SLM 계산을 단순화하려고 문맥 범위를 제한한 버전

- 한계
  - 희소 문제 여전함: N 키우면 조합 급증 → 데이터에 없는 조합 많아짐 → 확률 0
  - 문맥 제한: N-1개만 보니까 멀리 떨어진 단어 관계 파악 x
  - N 키우면 희소성, N 줄이면 문맥 부족 → 딜레마

### PPL (Perplexity)
- 모델이 문장 예측할 때 얼마나 **헷갈려하는지(perplexed)** 를 숫자로 나타낸 것
- **낮을수록 좋음** (덜 헷갈림 = 실제 단어 잘 맞춤)
- 희소한 문장일수록 PPL 높아지는 경향
- 단어 개수 늘어난다고 무조건 낮아지는 건 x. 추가된 단어가 쉬우면 낮아지고 어려우면 아님
- SLM, NLM 둘 다 평가할 때 씀
  - ex) 같은 데이터에서 SLM 80, NLM 25 → NLM이 훨씬 덜 헷갈림

### NLM vs SLM

NLM: SLM 한계 극복하려고 신경망으로 단어 의미랑 문맥 학습하는 모델
- 핵심 키워드: **임베딩**
  - SLM은 단어를 그냥 별개 기호로 취급
  - NLM은 단어를 벡터 공간의 한 점으로 표현 → 비슷한 애들끼리 가까이 모임
  - moon, mars, nasa는 우주동네 / bible, god, church는 종교동네

| 구분 | SLM | NLM |
|---|---|---|
| 학습 | 등장 빈도 세서 확률 계산 | 신경망 가중치 학습 |
| 단어 표현 | 개별 기호 | 연속 벡터 |
| 의미 관계 | 반영 어려움 | 임베딩으로 유사성 학습 |
| 희소 문제 | 안 본 조합에 취약 | 완화됨 (일반화 ㄱㄴ) |
| 문맥 | N-gram은 N-1개만 | 구조 따라 다양 |
| 비용 | 낮음 | 연산 많이 필요 |

→ NLM 장점: 의미 관계 학습 + 희소 문제 완화 + RNN, LSTM, 트랜스포머로 긴 문맥도 처리 ㄱㄴ

# 딥러닝 기반 모델

트랜스포머 복습 (4차시)
- RNN, LSTM 한계: 장기 의존성, 기울기 소실, 순차처리라 느림
- 트랜스포머: self-attention으로 장거리 문맥 잡고 병렬처리 ㄱㄴ
- 여기서 **인코더만 쌓으면 BERT, 디코더만 쌓으면 GPT**

### BERT
- 트랜스포머 **인코더** 기반, **양방향** 문맥 학습해서 언어를 "이해"하는 모델
  - B: Bidirectional (앞뒤 다 봄)
  - ER: Encoder Representations (인코더로 문맥 반영된 벡터 만듦)
  - T: from Transformers
- 사전학습 + 파인튜닝
  - 사전학습: 대량 텍스트로 언어 일반 구조 배움. 정답 라벨 필요 x
  - 파인튜닝: 특정 작업 데이터로 추가 학습

- 사전학습 방법 2개
  1. **MLM (Masked Language Model)**: 빈칸 채우기
     - 일부 토큰을 [MASK]로 가리고 앞뒤 문맥 보고 맞추기
     - "How are [MASK] doing today" → you
     - 앞뒤 다 보니까 양방향 이해력 up
  2. **NSP (Next Sentence Prediction)**: 문장 B가 문장 A 바로 다음 문장 맞음?
     - IsNext / NotNext 분류
     - 문장 간 관계 학습용

- 입력 처리: 토큰화 → [CLS], [SEP] 특수토큰 붙임 → 임베딩(의미+위치+문장구분) → 인코더 여러 층
- 이후 모델
  - RoBERTa: NSP 빼고 Dynamic Masking, 데이터 더 많이 → 성능 up
  - ALBERT: 파라미터 공유로 경량화, NSP 대신 SOP(문장 순서 예측)

### GPT
- 트랜스포머 **디코더** 기반, 이전 단어 보고 다음 단어 예측해서 **생성**하는 모델
- 인코더 없으니까 cross-attention도 x
- **Masked Self-Attention**이 핵심: 미래 토큰 못 봄 (컨닝 방지 그거)
- 자기회귀 방식: 하나 만들고 → 그거 붙여서 다음 거 만들고 → 반복
- GPT-3부터 파인튜닝 없이 프롬프트만으로 작업 수행 ㄱㄴ (In-Context Learning)
  - Zero-shot: 예시 없이 지시만
  - One-shot: 예시 1개
  - Few-shot: 예시 여러 개

| 구분 | BERT | GPT |
|---|---|---|
| 구조 | 인코더 | 디코더 |
| 방향 | 양방향 | 단방향(자기회귀) |
| 학습 | MLM, NSP | 다음 토큰 예측 |
| 목적 | 이해 | 생성 |
| 주 작업 | 감성분석, 분류, 개체명 인식 | 챗봇, 생성, 번역, 요약 |

BERT = 읽기 잘하는놈, GPT = 쓰기 잘하는놈

### LLM
- 방대한 데이터 + 대규모 신경망으로 언어 이해하고 생성하는 모델
- ex) GPT 계열, LLaMA, Claude, Gemini
- 대부분 트랜스포머 기반

- 작동
  - 학습: 사전학습(다음 토큰 예측 무한반복) + 파인튜닝
  - 예측: 입력 토큰화 → attention으로 관계 분석 → 다음 토큰 확률 계산 → 제일 높은 거 순서대로 뽑아서 답변

- LLM은 왜 GPT 방식? 
  - BERT는 이해 특화, GPT는 생성 특화
  - 대화형 LLM은 새 문장 만들어야 하니까 자기회귀 GPT 구조가 맞음

- 한계
  - **환각(Hallucination)**: 없는 정보를 그럴듯하게 지어냄. 확률로 다음 토큰 뽑는 거라 사실 여부 보장 x
  - 학습, 업데이트 비용 미쳤음
  - 학습 이후 정보 모름 → 외부 지식 끌어오는 기술 필요 → RAG

### RAG
- 외부 지식 **검색** + LLM **생성** 합친 것 → 정확성, 최신성 up
- 왜 만듦? LLM은 학습 당시 지식만 있어서 그 뒤 정보 답변 x

- 구조
  1. **질의 인코더 (Query Encoder)**: 질문을 임베딩해서 벡터로 바꿈
  2. **지식 검색기 (Knowledge Retriever)**: Vector DB에서 질문 벡터랑 비슷한 문서 찾음
  3. **지식 증강 생성기 (Knowledge-Augmented Generator)**: 질문 + 찾은 문서 같이 LLM에 넣어서 답변 생성

- 장점
  - 재학습 없이 최신 정보 반영 ㄱㄴ
  - 근거 있는 자료 기반이라 환각 완화
- 한계: 찾은 문서가 관련은 있는데 오래됐거나 틀릴 수 있음
  - → **CAG (Credibility-aware Generation)**: 관련성뿐 아니라 출처 신뢰도, 최신성까지 평가해서 신뢰도 높은 정보 위주로 답변
  - David Cameron 예시: RAG는 옛날 문서 보고 "공직 없음" 틀림, CAG는 최신 Reuters 보고 "외무장관" 맞춤

RAG = 오픈북 시험인데, CAG는 그 책이 믿을만한 책인지까지 따지는 느낌

### LangChain
- LLM을 외부 데이터, API, 도구랑 연결해서 AI 앱 쉽게 만들게 해주는 오픈소스 프레임워크
- 아이디어: LLM, RAG 직접 짜면 구현 복잡 + 외부 연동 귀찮 + 유지보수 지옥 → **기능을 모듈화해서 레고처럼 연결하자**

- 특징 3개
  1. **추상화**: 복잡한 API 호출 코드 대신 간단한 명령어로 LLM 사용
  2. **표준화**: 모델 바꿔도 연결 부분만 갈아끼우면 됨 (ChatGPT → Claude)
  3. **체이닝**: 작업을 순서대로 연결 (번역 → 요약 → 출력)

- 한계: 체이닝은 정해진 순서대로만 가는 선형 구조
  - 결과 따라 분기하거나 이전 단계로 돌아가야 하면? → **LangGraph**
  - 노드(작업 단위) + 엣지(연결, 경로) + 상태(공유 정보)로 분기, 반복 ㄱㄴ
- LangSmith: 실행 추적, 디버깅, 평가, 모니터링, 프롬프트 버전관리 하는 관리 플랫폼

(API 키는 처음 한 번만 보임 + 절대 공유 x. 내 지갑에서 돈 나감)

# sLM (small Language Model)

- LLM보다 규모 작은 언어모델
  - sLM: 수백만~수십억 파라미터 / LLM: 수십억~수천억 이상
- 작아서 메모리, 연산 적게 씀 → 폰 같은 데서도 돌아감, 인터넷 없이 오프라인 추론도 ㄱㄴ
- ex) DistilBERT, Gemma, GPT-4o mini, Llama 소형, Ministral, Phi

### 모델 압축
**성능은 최대한 유지하면서 크기랑 연산량 줄이는 기술**

1. **가지치기 (Pruning)**: 별로 안 중요한 연결이나 뉴런 잘라냄
   - 비구조적(연결만 제거) / 구조적(뉴런, 블록 통째로)
   - 중요한 것까지 자르면 성능 떨어짐
2. **양자화 (Quantization)**: 숫자 정밀도 낮춤
   - FP32 → FP16, INT8
   - 비트 수 줄어서 가벼워짐. 대신 정밀도 손해
3. **저차원 분해 (Low-Rank Factorization)**: 큰 행렬을 작은 행렬 두 개 곱으로 근사
   - LoRA가 이 원리 씀 (근데 LoRA는 모델 압축이 아니라 변경량만 작은 행렬로 학습하는 미세조정 방법)
4. **지식 증류 (Knowledge Distillation)**: 큰 모델(선생님)의 지식을 작은 모델(학생)한테 전수
   - 학생이 선생님 출력 따라하면서 배움
   - DistilBERT가 이거

### sLM 장단점
- 장점
  - 성능 측면: 빠른 응답, 제한된 환경에서도 실행, 내부에서 돌리니까 데이터 밖으로 안 나감(보안)
  - 비용 측면: 운영비 싸고, 전기 덜 먹고, 비싼 GPU 의존도 낮음
- 단점
  - 편향: 데이터 다양성 부족하면 한쪽으로 치우침
  - 일반화, 추론 능력 약함: 복잡하거나 전문적인 작업에서 LLM보다 정확도 떨어짐
  - 환각: sLM도 지어냄

### sLM + LLM 결합 (Hybrid Inference)
- 쉬운 일은 sLM, 어려운 일은 LLM
  - 문장 분류, 간단 요약 → sLM이 빠르게
  - 여러 단계 추론, 전문 분석 → LLM이
- 모든 걸 LLM으로 안 돌려도 되니까 비용, 시간 절약 + 필요한 성능 확보
- 챗봇에서 간단한 질문은 sLM이 바로 답하고 복잡한 분석 요청은 LLM한테 넘기는 식

동네 의원(sLM)이랑 대학병원(LLM) 느낌. 감기는 동네에서, 큰 수술은 대학병원에서

# 분류 모델 vs 생성 모델

생성 모델: 학습 데이터의 **분포**를 배워서 그 분포를 따르는 비슷한 새 데이터 만드는 모델

| 구분 | 분류 모델 | 생성 모델 |
|---|---|---|
| 학습 목표 | P(Y\|X): X 주어지면 정답 Y일 확률 | P(X): 데이터 X 자체의 분포 |
| 학습하는 것 | 데이터 간 **차이점** → 경계선 | 데이터의 **공통 특징, 패턴** |
| 출력 | 범주, 범주별 확률 | 새 이미지, 텍스트, 음성 |
| 한계 | 처음 보는 이상한 데이터도 아는 라벨 중 하나로 우겨넣음, 생성 x | 부자연스러운 결과 나올 수 있음, 품질 평가 어려움 |

분류: "이거 강아지야 고양이야?" / 생성: "강아지란 이렇게 생긴 거구나 → 하나 그려볼게"

# 생성 모델의 분류

기준: **확률밀도 P(X)를 직접 계산할 수 있냐?**
- 확률밀도: 이 데이터가 학습한 분포에서 얼마나 그럴듯한지 (강아지 학습했으면 강아지는 높고 자동차는 낮음)

1. **명시적 (Explicit)**: P(X) 수치로 계산하거나 근사 ㄱㄴ
   - 다루기 쉬운 밀도 (Tractable): 직접 정확히 계산 (PixelRNN/CNN 등)
   - 근사적 밀도 (Approximate): 계산 어려워서 근사함
     - VAE: 변분 추론 (복잡한 분포를 정규분포 같은 쉬운 분포로 근사)
     - Boltzmann Machine: 마르코프 연쇄 (다음 상태가 현재 상태로만 결정됨. 오늘 날씨 → 내일 날씨)
2. **암시적 (Implicit)**: P(X) 안 알려줌. 그냥 새 데이터 뽑아내는 데 집중
   - 직접 샘플링: 노이즈 넣고 한 번에 생성 → GAN
   - 간접/반복 샘플링: 상태 변화 여러 번 반복해서 생성 → GSN

### AE (Auto Encoder)
- 입력을 압축했다가 최대한 원래대로 복원하도록 학습하는 신경망
- 정답 라벨 필요 x. 입력 자체가 정답
- 구조: 인코더 → 병목(Bottleneck) → 디코더
  - 인코더: 입력 x에서 중요한 특징 뽑아서 잠재벡터 z로 압축
  - 병목: 좁은 공간에 넣으니까 핵심만 남기게 됨
  - 디코더: z로 x' 복원
- 기존 seq2seq 인코더디코더랑 차이: 걔는 번역같이 다른 걸 만들고, AE는 **자기 자신 복원**이 목적 (4 넣으면 4 나옴)

- 왜 압축했다 복원함?
  - 압축 안 하면 그냥 복사해버림 → 아무것도 안 배움
  - 공간 좁히면 뭐가 중요한지 골라야 함 → 숫자 4의 획, 형태 같은 핵심 구조 학습
  - 복원은 압축이 잘 됐는지 확인용 (복원 오차 줄이기)
- 활용: 데이터 압축, 노이즈 제거, 이상 탐지

### VAE (Variational Auto Encoder)
- 왜 만듦? AE의 잠재공간이 불규칙함
  - 데이터 없는 빈 영역에서 z 뽑아서 디코더 넣으면 이상한 거 나옴
  - AE는 복원은 잘하는데 **새로운 거 생성은 못함**
- 그래서 VAE: 데이터를 **점 하나가 아니라 확률분포로** 표현

- AE vs VAE
  - AE: 특징을 고정된 점으로 → 잠재공간 듬성듬성
  - VAE: 평균(μ), 표준편차(σ)로 분포 표현 → 잠재공간이 연속적이고 규칙적 → 아무데서나 뽑아도 자연스러움

- 학습 목표 2개
  1. 복원 오차 최소화: 원본이랑 비슷하게 복원
  2. KL 발산 최소화: 잠재분포가 표준정규분포랑 비슷해지게 (KL 발산 = 두 분포가 얼마나 다른지)

- 작동 과정
  1. 분포 예측 (확률적 인코더): x 넣고 μ, σ 예측
  2. 샘플링 + 재파라미터화: z = μ + σ⊙ε (ε은 표준정규분포 노이즈)
     - 그냥 랜덤 샘플링하면 기울기가 안 흘러서 학습 x → 랜덤성을 ε로 따로 빼버림
     - μ=2, σ=0.5일 때 ε=1이면 z=2.5, ε=-1이면 z=1.5 → 같은 분포에서도 다른 z
  3. 생성 (확률적 디코더): z로 데이터 만듦
     - 학습 끝나면 표준정규분포에서 z 뽑아서 디코더에 넣으면 학습 데이터에 없던 새로운 4 생성 ㄱㄴ

AE는 복원기, VAE는 생성기로 진화

### GAN (Generative Adversarial Network)
- 생성자(G)랑 판별자(D)가 서로 싸우면서 진짜 같은 데이터 만드는 모델 (2014, Ian Goodfellow)
- 비유: 위조지폐범(생성자) vs 경찰(판별자)
  - 범인은 더 정교한 위조지폐 만들고, 경찰은 더 잘 잡아내고
  - 끝까지 가면 진짜랑 구분 x → 판별자가 찍어서 맞추는 수준(0.5)

- 구조
  - 무작위 노이즈 z → **생성자** → 가짜 이미지 G(z)
  - 진짜 x or 가짜 G(z) → **판별자** → 진짜일 확률 (0~1)
  - 판별자: 진짜엔 1, 가짜엔 0 주려고 함 → **최대화**
  - 생성자: 판별자 속여서 가짜에도 1 받으려고 함 → **최소화**
  - 이게 최소-최대 게임 (Minimax Game)
- 학습 끝나면 생성자만 씀

- 장점
  - 진짜 같은 가짜 (실존 안 하는 사람 얼굴)
  - 선명함: VAE는 픽셀 평균(MSE) 최적화라 흐릿한데, GAN은 판별자 속여야 해서 질감까지 살림
  - 빠름: 노이즈 한 번 통과하면 끝 (확산모델은 여러 번 반복)
  - 잠재공간 부드러움 (z 조금씩 바꾸면 표정이 서서히 바뀜. 근데 항상 보장 x)

- 단점
  - **학습 불안정**: 한쪽이 너무 세지면 균형 깨짐 → 진동, 수렴 못하고 무한루프
    - 해결: 학습률/횟수 조절, WGAN-GP
  - **훈련 초기 문제**: 처음엔 생성자가 너무 못해서 판별자가 바로 잡아냄 → 기울기 거의 0 → 생성자가 뭘 고쳐야 할지 모름
    - 해결: Non-saturating Loss
  - **모드 붕괴**: 판별자 잘 속인 한 가지 스타일만 계속 만듦 (똑같이 생긴 강아지만 무한생성)
    - 해결: Minibatch Discrimination, WGAN-GP
  - **평가 어려움**: 사실성, 선명도, 다양성을 한 지표로 못 잼 → FID, IS + 사람 눈
  - **역매핑 어려움**: z → 이미지는 되는데 이미지 → z는 못함 (인코더가 없어서)
    - 해결: GAN Inversion, BiGAN
- 적용 사례: NVIDIA 가상인물(Progressive GAN), 인물 영상 합성(cGAN), 눈 감은 사진 눈 뜨게(ExGAN)

### 확산 모델 (Diffusion Model)
- 이미지에 노이즈 넣는 과정을 보고 → 노이즈 빼는 법을 배워서 → 순수 노이즈에서 새 이미지 만드는 모델
- 목적: 데이터 분포 학습해서 자연스럽고 **다양한** 데이터 생성
  - 모호성 모델링: 노이즈 낀 이미지는 원래 뭐였는지 하나로 못 정함 → 가능한 이미지들의 분포를 학습
  - 같은 "공원에 앉은 강아지"여도 품종, 털색, 자세 다 다르게 나옴

#### 순확산 (Forward Diffusion)
- 원본 x0에 가우시안 노이즈(정규분포 따르는 잡음)를 조금씩 추가
- x0 → x1 → x2 → ... → xT
- 단계 갈수록 바다랑 절벽이 흐려지다가 결국 그냥 노이즈 화면
- 목표: 다양한 노이즈 단계의 데이터를 만들어서 역확산 학습 재료로 씀

#### 역확산 (Reverse Diffusion)
- 반대로 무작위 노이즈 xT에서 시작해서 노이즈를 단계적으로 제거
- Denoising U-Net이 지금 이미지에 낀 노이즈를 예측하고 조금씩 뺌
- 반복하다 보면 색상, 윤곽 나타나고 → 최종 새 이미지 x0

순확산 = 망가뜨리는 과정 (학습용 데이터 만들기), 역확산 = 복구하는 과정 (실제 생성)

- 장점
  - 품질 압도적: 한 번에 만드는 GAN보다 단계적으로 다듬어서 더 정교함
  - 학습 안정: 경쟁 구조 x. 실제 노이즈랑 예측 노이즈 차이(MSE)만 줄이면 됨 → 손실 곡선으로 확인 ㄱㄴ
  - 다양성: 랜덤 노이즈에서 시작하니까 결과 매번 다름 (GAN 모드붕괴랑 대비)
- 단점
  - 느림: 노이즈 제거를 여러 번 반복해야 함 (GAN은 한 방)
  - 비쌈: 생성할 때도 신경망 반복 실행 → 고성능 GPU 필요
  - 구조적 제약: 단계 수나 방식에 따라 속도, 품질 달라져서 설계 신경 써야 함


### 생성모델 한 줄 정리
- AE: 압축 → 복원. 생성은 잘 못함
- VAE: 분포로 압축해서 생성 ㄱㄴ. 근데 흐릿함
- GAN: 경쟁으로 선명하고 빠름. 근데 학습 불안정, 모드붕괴
- 확산: 품질, 다양성, 안정성 다 좋음. 근데 느리고 비쌈